# 🌐 크로스섹셔널 멀티종목 주가 방향성 예측 모델

`US_final_predict_v2.ipynb`(NVDA 단일 종목 + 피어 풀링)에서 한 단계 더 나아가,
**8개 섹터 ~27개 대형주를 하나의 모델로 학습**하는 진짜 "범용" 모델입니다.

## v2와의 핵심 차이
- **단일 종목 시계열 → 크로스섹셔널 패널**: "이 종목이 오를까?"가 아니라
  "오늘 유니버스 안에서 어떤 종목이 상대적으로 더 오를 것 같은가?"를 학습
- **정규화 방식**: 종목별 시계열 스케일링(RobustScaler) 대신, **날짜별 횡단면 z-score** 사용
  (그날 유니버스 평균/표준편차 기준으로 정규화 → 종목 간 직접 비교 가능)
- **분할 방식**: 종목별 비율 분할 대신 **절대 날짜 컷오프** (상장일이 제각각이라 비율 분할은 부적절)
- **PatchTST(Transformer) 제외**: v2에서 검증 가중치가 0.005로 사실상 기여가 없었음.
  LightGBM + RandomForest 조합에 집중 (필요하면 v2 구조를 참고해 다시 추가 가능)
- **평가/백테스트도 재설계**: 단일 종목 타이밍 → 매일 유니버스 중 상위 K종목 롱 포트폴리오,
  실전 대시보드도 단일 종목이 아닌 **전체 유니버스 랭킹 테이블**

## 이 모델의 장점
- 학습 데이터가 종목당이 아니라 유니버스 전체로 늘어나 특정 종목 과적합 위험이 줄어듦
- 학습에 없던 종목도(같은 방식으로 피처만 만들면) 원리상 예측 가능 — 진짜 범용 모델
- 종목별 Test AUC를 따로 확인해서 특정 섹터에 편중되지 않았는지 점검 가능

## 주의
- GPU 불필요 (트리 모델만 사용, CPU로 충분)
- 27개 종목 + 거시경제 지수를 매번 새로 받아오므로 실행 시간은 v2와 비슷하거나 약간 김
- 크로스섹셔널 정규화 특성상, 실전 예측 시에도 **유니버스 전체 데이터가 있어야** 정확한 정규화가 가능합니다 (단일 종목만 넣으면 상대 비교 기준이 없음)

In [1]:
# =====================================================================
# ⚙️ Cell 1: 설정값 지정 (크로스섹셔널 멀티종목)
# =====================================================================
import random
import datetime
import pickle
import os
import numpy as np
import pandas as pd
import yfinance as yf
import lightgbm as lgb
import optuna
from optuna.samplers import TPESampler          # ✅ 재현성: 샘플러 시드 고정용
optuna.logging.set_verbosity(optuna.logging.WARNING)
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy.optimize import minimize
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV   # ✅ #15 확률 캘리브레이션
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix, roc_curve
from ta.momentum import RSIIndicator
from ta.trend import SMAIndicator, EMAIndicator, MACD
from ta.volatility import AverageTrueRange, BollingerBands
from ta.volume import OnBalanceVolumeIndicator

try:
    font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    font_prop = fm.FontProperties(fname=font_path)
    plt.rcParams['font.family'] = font_prop.get_name()
except:
    plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# =====================================================================
# 🌐 유니버스: 8개 섹터 대형주
# =====================================================================
# ⚠️ [생존 편향 주의] 아래 유니버스는 '현재 시점'의 성공한 대형주로 구성됨.
#    2015년부터 백테스트하면 파산·상장폐지·몰락 종목이 배제되어 초과수익이
#    구조적으로 부풀려질 수 있음. 실전 판단 시 이 한계를 반드시 감안할 것.
#    (정석: 각 시점의 실제 지수 구성종목 이력(point-in-time universe) 사용)
UNIVERSE = [
    # 기술
    'AAPL', 'MSFT', 'GOOGL', 'NVDA', 'META', 'AVGO', 'ORCL',
    # 금융
    'JPM', 'BAC', 'GS', 'V', 'MA',
    # 헬스케어
    'UNH', 'JNJ', 'LLY', 'PFE',
    # 임의소비재
    'AMZN', 'TSLA', 'HD', 'MCD',
    # 필수소비재
    'PG', 'KO', 'WMT',
    # 에너지
    'XOM', 'CVX',
    # 산업재
    'CAT', 'BA',
    # 커뮤니케이션서비스
    'DIS', 'NFLX',
]
print(f'🌐 유니버스: {len(UNIVERSE)}개 종목 (8개 섹터)')

START_DATE = '2015-01-01'
END_DATE   = datetime.datetime.today().strftime('%Y-%m-%d')

# ✅ 절대 날짜 기준 분할 (종목마다 상장/데이터 길이가 달라 비율 분할 대신 날짜 컷오프 사용)
TRAIN_END_DATE = '2024-01-01'
VAL_END_DATE   = '2025-01-01'
# TEST 구간: VAL_END_DATE ~ END_DATE

PRED_DAYS = 3
THRESHOLD_CANDIDATES = [0.003, 0.005, 0.008, 0.012, 0.017, 0.025]

# ✅ 백테스트/전략 파라미터 (앞단에서 미리 정의 → threshold '경제적' 선택에도 재사용)
TOP_K            = 5       # 매 리밸런스 상위 K종목 동일비중 롱
TRANSACTION_COST = 0.001   # ✅ #4 편도 거래비용 0.1% (왕복 0.2%) — 수수료+슬리피지 근사

# 가중치 초기값 (뒤에서 Validation AUC 기준으로 자동 최적화되어 덮어씀)
W_LGBM = 0.5
W_RF   = 0.5

print(f'📅 Train : {START_DATE} ~ {TRAIN_END_DATE}')
print(f'📅 Val   : {TRAIN_END_DATE} ~ {VAL_END_DATE}')
print(f'📅 Test  : {VAL_END_DATE} ~ {END_DATE}')
print(f'🔮 예측 타겟: 미래 {PRED_DAYS}영업일 뒤 방향성')
print(f'🎯 Threshold 후보: {THRESHOLD_CANDIDATES}')
print(f'💸 거래비용(편도): {TRANSACTION_COST*100:.2f}% | Top-K={TOP_K} | SEED={SEED}')

c:\Users\smhrd\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🌐 유니버스: 29개 종목 (8개 섹터)
📅 Train : 2015-01-01 ~ 2024-01-01
📅 Val   : 2024-01-01 ~ 2025-01-01
📅 Test  : 2025-01-01 ~ 2026-07-23
🔮 예측 타겟: 미래 3영업일 뒤 방향성
🎯 Threshold 후보: [0.003, 0.005, 0.008, 0.012, 0.017, 0.025]
💸 거래비용(편도): 0.10% | Top-K=5 | SEED=42


In [2]:
# =====================================================================
# 📥 Cell 2: 데이터 수집 + 피처 엔지니어링 함수
# =====================================================================

def fetch_us_data(ticker, start, end):
    # ✅ #8 auto_adjust=True: 분할·배당 반영된 수정주가 사용.
    #    (미지정 시 yfinance 버전에 따라 결과가 달라져 재현성이 깨지고,
    #     분할일에 Return이 -90%로 잡혀 라벨/피처가 오염됨 — NVDA 10:1 등)
    df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
    df.index = pd.to_datetime(df.index)
    df.dropna(inplace=True)
    return df

def fetch_external_data(start, end):
    print('📡 글로벌 거시경제 지수 동기화 중...')
    tickers = {
        'SP500' : '^GSPC',
        'NDX'   : '^NDX',
        'VIX'   : '^VIX',
        'DXY'   : 'DX-Y.NYB',
        'OIL'   : 'CL=F',
        'TNX'   : '^TNX',
    }
    ext = pd.DataFrame()
    for name, sym in tickers.items():
        try:
            data = yf.download(sym, start=start, end=end, progress=False, auto_adjust=True)
            if isinstance(data.columns, pd.MultiIndex):
                data.columns = data.columns.get_level_values(0)
            ext[name] = data['Close']
        except Exception as e:                       # ✅ bare except → 원인 로깅
            print(f'  ⚠️ {name} 다운로드 실패: {e}')
    ext.index = pd.to_datetime(ext.index)
    return ext

def add_features(df, threshold, pred_days, for_predict=False):
    """for_predict=False : 학습용 — Future_Return/Label 생성 (마지막 pred_days행은 라벨이 없어 dropna로 제거).
       for_predict=True  : 실전 예측용 — 라벨을 만들지 않아 '가장 최근 거래일'까지 보존.
                           ⚠️ 예측 경로에서 학습용(False)을 쓰면 최신 pred_days일이 잘려나가
                              사실상 pred_days일 전 데이터로 예측하는 버그가 생김(#1)."""
    d = df.copy()

    if 'SP500' in d.columns: d['SP500_return_1d'] = d['SP500'].pct_change().shift(1)
    if 'NDX' in d.columns:   d['NDX_return_1d']   = d['NDX'].pct_change().shift(1)
    if 'DXY' in d.columns:   d['DXY_return_1d']   = d['DXY'].pct_change().shift(1)
    if 'OIL' in d.columns:   d['OIL_return_1d']   = d['OIL'].pct_change().shift(1)
    if 'VIX' in d.columns:   d['VIX_diff_1d']     = d['VIX'].diff().shift(1)
    if 'TNX' in d.columns:   d['TNX_diff_1d']     = d['TNX'].diff().shift(1)

    d['Return'] = d['Close'].pct_change()
    for i in [2, 5, 10, 20]: d[f'Return_{i}d'] = d['Close'].pct_change(i)
    d['HL_ratio'] = (d['High'] - d['Low']) / d['Close']
    d['OC_ratio'] = (d['Close'] - d['Open']) / (d['Open'] + 1e-9)
    d['Gap'] = (d['Open'] - d['Close'].shift(1)) / (d['Close'].shift(1) + 1e-9)

    for w in [5, 10, 20, 60, 120]:
        sma = SMAIndicator(d['Close'], window=w).sma_indicator()
        d[f'SMA_{w}'] = sma
        d[f'SMA_dist_{w}'] = (d['Close'] - sma) / (sma + 1e-9)

    d['EMA_12'] = EMAIndicator(d['Close'], window=12).ema_indicator()
    d['EMA_26'] = EMAIndicator(d['Close'], window=26).ema_indicator()
    macd = MACD(d['Close'])
    d['MACD'], d['MACD_signal'], d['MACD_diff'] = macd.macd(), macd.macd_signal(), macd.macd_diff()

    for w in [7, 14, 21]: d[f'RSI_{w}'] = RSIIndicator(d['Close'], window=w).rsi()

    d['Volume_ratio'] = d['Volume'] / (d['Volume'].rolling(20).mean() + 1e-9)
    d['Vol_5d']  = d['Return'].rolling(5).std()
    d['Vol_20d'] = d['Return'].rolling(20).std()
    d['Vol_ratio'] = d['Vol_5d'] / (d['Vol_20d'] + 1e-9)

    if 'NDX_return_1d' in d.columns:
        d['vs_NDX_1d'] = d['Return'].shift(1) - d['NDX_return_1d']

    raw_macro_cols = ['SP500', 'NDX', 'VIX', 'DXY', 'OIL', 'TNX']
    d = d.drop(columns=[c for c in raw_macro_cols if c in d.columns], errors='ignore')

    if not for_predict:
        d['Future_Return'] = (d['Close'].shift(-pred_days) - d['Close']) / d['Close']
        d['Label'] = (d['Future_Return'] > threshold).astype(int)

    d.replace([np.inf, -np.inf], np.nan, inplace=True)
    # ✅ #2 bfill 제거: bfill은 미래값을 과거로 역채움하는 look-ahead 누출.
    #    ffill만 적용하고, 지표 워밍업(SMA_120 등)으로 생긴 앞쪽 NaN은 dropna로 제거.
    d = d.ffill()
    if for_predict:
        # 라벨/미래수익 컬럼은 존재하지 않으므로 피처 기준으로만 결측 제거 → 최신 거래일 보존
        return d.dropna()
    return d.dropna()

print('✅ 데이터 수집 / 피처 엔지니어링 함수 정의 완료 (auto_adjust·for_predict·bfill제거 반영)')

✅ 데이터 수집 / 피처 엔지니어링 함수 정의 완료 (auto_adjust·for_predict·bfill제거 반영)


In [3]:
# =====================================================================
# 📡 Cell 3: 유니버스 전체 원본 데이터 수집 (한 번만 다운로드해서 캐시)
# =====================================================================
print(f'📡 거시경제 지수 수집 중...')
df_ext = fetch_external_data(START_DATE, END_DATE)

raw_data = {}
failed = []
for i, tk in enumerate(UNIVERSE, 1):
    try:
        df_raw = fetch_us_data(tk, START_DATE, END_DATE)
        df_merged = df_raw.join(df_ext, how='left').ffill().dropna()
        raw_data[tk] = df_merged
        print(f'  ✅ [{i}/{len(UNIVERSE)}] {tk}: {len(df_merged)}일치')
    except Exception as e:
        failed.append(tk)
        print(f'  ⚠️ [{i}/{len(UNIVERSE)}] {tk} 실패: {e}')

if failed:
    print(f'\n⚠️ 수집 실패 종목 (유니버스에서 제외): {failed}')
    UNIVERSE = [t for t in UNIVERSE if t not in failed]

print(f'\n✅ {len(UNIVERSE)}개 종목 원본 데이터 수집 완료')

📡 거시경제 지수 수집 중...
📡 글로벌 거시경제 지수 동기화 중...
  ✅ [1/29] AAPL: 2904일치
  ✅ [2/29] MSFT: 2904일치
  ✅ [3/29] GOOGL: 2904일치
  ✅ [4/29] NVDA: 2904일치
  ✅ [5/29] META: 2904일치
  ✅ [6/29] AVGO: 2904일치
  ✅ [7/29] ORCL: 2904일치
  ✅ [8/29] JPM: 2904일치
  ✅ [9/29] BAC: 2904일치
  ✅ [10/29] GS: 2904일치
  ✅ [11/29] V: 2904일치
  ✅ [12/29] MA: 2904일치
  ✅ [13/29] UNH: 2904일치
  ✅ [14/29] JNJ: 2904일치
  ✅ [15/29] LLY: 2904일치
  ✅ [16/29] PFE: 2904일치
  ✅ [17/29] AMZN: 2904일치
  ✅ [18/29] TSLA: 2904일치
  ✅ [19/29] HD: 2904일치
  ✅ [20/29] MCD: 2904일치
  ✅ [21/29] PG: 2904일치
  ✅ [22/29] KO: 2904일치
  ✅ [23/29] WMT: 2904일치
  ✅ [24/29] XOM: 2904일치
  ✅ [25/29] CVX: 2904일치
  ✅ [26/29] CAT: 2904일치
  ✅ [27/29] BA: 2904일치
  ✅ [28/29] DIS: 2904일치
  ✅ [29/29] NFLX: 2904일치

✅ 29개 종목 원본 데이터 수집 완료


In [4]:
# =====================================================================
# ⚙️ Cell 4: 패널(long-format) 구성 + 크로스섹셔널 정규화 + 날짜 분할 함수
#            + 성과지표/백테스트 공용 헬퍼 (#4 거래비용 · #10 Sharpe/MDD)
# =====================================================================

EXCLUDE_COLS = ['Date', 'Ticker', 'Label', 'Open', 'High', 'Low', 'Close', 'Volume', 'Future_Return']

def build_panel(threshold):
    """유니버스 전체 종목에 피처를 생성한 뒤 하나의 패널(long-format) 데이터프레임으로 결합"""
    frames = []
    for tk in UNIVERSE:
        df_feat = add_features(raw_data[tk], threshold=threshold, pred_days=PRED_DAYS)
        df_feat['Ticker'] = tk
        frames.append(df_feat)
    panel = pd.concat(frames, axis=0)
    panel.index.name = 'Date'
    panel = panel.reset_index()
    return panel

def cross_sectional_normalize(panel, feature_cols, clip=5.0):
    """✅ [핵심] 같은 날짜의 여러 종목끼리 서로 비교 가능하도록,
    날짜별로 각 피처를 평균 0 / 표준편차 1로 정규화(z-score).
    → 절대적인 수치가 아니라 '그날 유니버스 내 상대적 위치'를 학습하게 됨.
    (날짜별 독립 계산이라 미래 정보 누출 없음.) 극단값은 clip으로 완화."""
    grouped = panel.groupby('Date')[feature_cols]
    mean = grouped.transform('mean')
    std  = grouped.transform('std').replace(0, np.nan)
    z = (panel[feature_cols] - mean) / std
    z = z.clip(-clip, clip).fillna(0.0)
    result = panel.copy()
    result[feature_cols] = z
    return result

def split_by_date(panel):
    train_end = pd.Timestamp(TRAIN_END_DATE)
    val_end   = pd.Timestamp(VAL_END_DATE)
    train_mask = panel['Date'] < train_end
    val_mask   = (panel['Date'] >= train_end) & (panel['Date'] < val_end)
    test_mask  = panel['Date'] >= val_end
    return train_mask, val_mask, test_mask

# ---------------------------------------------------------------------
# 📊 성과지표 (#10) — 수익률 시계열 → Sharpe/Sortino/MDD/Calmar/승률/PF
# ---------------------------------------------------------------------
def perf_metrics(period_returns, pred_days=PRED_DAYS):
    """리밸런스 주기(pred_days)마다의 수익률 배열을 받아 핵심 지표를 반환.
    ppy: 연간 리밸런스 횟수(≈252/pred_days)로 연율화."""
    r = np.asarray(period_returns, dtype=float)
    out = {'cum': 0.0, 'sharpe': 0.0, 'sortino': 0.0, 'mdd': 0.0,
           'calmar': 0.0, 'winrate': 0.0, 'pf': 0.0, 'n': int(len(r))}
    if len(r) == 0:
        return out
    ppy = 252.0 / pred_days
    out['cum'] = float(np.prod(1 + r) - 1)
    mean = r.mean()
    sd = r.std(ddof=1) if len(r) > 1 else 0.0
    out['sharpe'] = float(mean / sd * np.sqrt(ppy)) if sd > 0 else 0.0
    downside = r[r < 0]
    dsd = downside.std(ddof=1) if len(downside) > 1 else 0.0
    out['sortino'] = float(mean / dsd * np.sqrt(ppy)) if dsd > 0 else 0.0
    eq = np.cumprod(1 + r)
    peak = np.maximum.accumulate(eq)
    out['mdd'] = float(((eq - peak) / peak).min())
    out['calmar'] = float(out['cum'] / abs(out['mdd'])) if out['mdd'] < 0 else 0.0
    out['winrate'] = float((r > 0).mean())
    gains = r[r > 0].sum(); losses = -r[r < 0].sum()
    out['pf'] = float(gains / losses) if losses > 0 else float('inf')
    return out

def print_metrics(name, m):
    print(f'   [{name}] 누적={m["cum"]*100:6.1f}% | Sharpe={m["sharpe"]:5.2f} | '
          f'Sortino={m["sortino"]:5.2f} | MDD={m["mdd"]*100:6.1f}% | '
          f'Calmar={m["calmar"]:5.2f} | 승률={m["winrate"]*100:4.1f}% | PF={m["pf"]:.2f} | n={m["n"]}')

# ---------------------------------------------------------------------
# 🚀 크로스섹셔널 Top-K 롱 백테스트 (#4 거래비용 반영)
# ---------------------------------------------------------------------
def crosssec_topk_backtest(panel_with_prob, top_k, pred_days,
                           cost=0.0, prob_col='pred_prob'):
    """매 리밸런스일 prob 상위 top_k 동일비중 롱. pred_days마다 리밸런스.
    반환: (전략수익 배열, 시장(유니버스 동일비중)수익 배열, 선택종목 리스트)
    비용: 진입+청산 왕복이므로 2*cost 차감."""
    dates = sorted(panel_with_prob['Date'].unique())
    port, mkt, picks = [], [], []
    i = 0
    while i < len(dates):
        d = dates[i]
        day = panel_with_prob[panel_with_prob['Date'] == d].sort_values(prob_col, ascending=False)
        topk = day.head(top_k)
        gross = topk['Future_Return'].mean() if len(topk) > 0 else 0.0
        port.append(gross - 2 * cost)   # 왕복 거래비용 차감
        mkt.append(day['Future_Return'].mean() if len(day) > 0 else 0.0)
        picks.extend(topk['Ticker'].tolist())
        i += pred_days
    return np.array(port), np.array(mkt), picks

print('✅ 패널/정규화/분할 + 성과지표(perf_metrics)·백테스트(crosssec_topk_backtest) 헬퍼 정의 완료')

✅ 패널/정규화/분할 + 성과지표(perf_metrics)·백테스트(crosssec_topk_backtest) 헬퍼 정의 완료


In [ ]:
# =====================================================================
# 🔍 Cell 5: Threshold 후보 탐색
#   ✅ #11 선택 기준을 'AUC'가 아니라 '검증구간 Top-K 전략 Sharpe(경제적 기준)'로 변경.
#      threshold를 바꾸면 라벨 정의 자체가 달라져 AUC 비교는 서로 다른 문제를 비교하는 꼴.
#      실제 목적(수익성)과 일치하도록 검증구간 백테스트 Sharpe로 고른다.
# =====================================================================
print('🔍 1단계: Threshold 후보 탐색 중 (기준: 검증구간 Top-K Sharpe)...')
print('='*70)

threshold_scores = {}   # {threshold: val_sharpe}
threshold_aux    = {}   # 참고용 AUC/누적수익
for t in THRESHOLD_CANDIDATES:
    panel_t = build_panel(threshold=t)
    feature_cols_t = [c for c in panel_t.columns if c not in EXCLUDE_COLS]
    panel_t_norm = cross_sectional_normalize(panel_t, feature_cols_t)

    tr_m, vl_m, te_m = split_by_date(panel_t_norm)
    up_ratio = panel_t_norm.loc[tr_m, 'Label'].mean()

    if not (0.35 <= up_ratio <= 0.65):
        print(f'   threshold={t:.3f} | 상승={up_ratio*100:.1f}% → ⚠️ 스킵 (불균형 심함)\n')
        continue

    X_tr_t = panel_t_norm.loc[tr_m, feature_cols_t].values
    y_tr_t = panel_t_norm.loc[tr_m, 'Label'].values
    X_vl_t = panel_t_norm.loc[vl_m, feature_cols_t].values
    y_vl_t = panel_t_norm.loc[vl_m, 'Label'].values

    lgb_temp = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05,
                                    class_weight='balanced', random_state=SEED, verbose=-1)
    lgb_temp.fit(X_tr_t, y_tr_t, eval_set=[(X_vl_t, y_vl_t)],
                 callbacks=[lgb.early_stopping(20, verbose=False)])
    lgb_val = lgb_temp.predict_proba(X_vl_t)[:, 1]

    rf_temp = RandomForestClassifier(n_estimators=200, max_depth=8,
                                       class_weight='balanced', random_state=SEED, n_jobs=-1)
    rf_temp.fit(X_tr_t, y_tr_t)
    rf_val = rf_temp.predict_proba(X_vl_t)[:, 1]

    avg_auc = (roc_auc_score(y_vl_t, lgb_val) + roc_auc_score(y_vl_t, rf_val)) / 2

    # 🎯 경제적 기준: 검증구간에서 Top-K 롱 전략을 돌려 Sharpe 계산
    #    (crosssec_topk_backtest가 'Ticker' 컬럼을 쓰므로 반드시 포함)
    val_bt = panel_t_norm.loc[vl_m, ['Date', 'Ticker', 'Future_Return']].copy()
    val_bt['pred_prob'] = 0.5 * lgb_val + 0.5 * rf_val
    port_v, _, _ = crosssec_topk_backtest(val_bt, TOP_K, PRED_DAYS, cost=TRANSACTION_COST)
    m_v = perf_metrics(port_v, pred_days=PRED_DAYS)

    threshold_scores[t] = m_v['sharpe']
    threshold_aux[t]    = (avg_auc, m_v['cum'])
    print(f'   threshold={t:.3f} | 상승={up_ratio*100:.1f}% | AUC={avg_auc:.4f} | '
          f'Val누적={m_v["cum"]*100:5.1f}% | Val Sharpe={m_v["sharpe"]:.3f}')

if not threshold_scores:
    raise RuntimeError('모든 threshold 후보가 스킵됨 — THRESHOLD_CANDIDATES 범위를 조정하세요.')

BEST_THRESHOLD = max(threshold_scores, key=threshold_scores.get)
_auc, _cum = threshold_aux[BEST_THRESHOLD]
print('='*70)
print(f'\n✅ 최적 Threshold 확정: {BEST_THRESHOLD:.3f} '
      f'(Val Sharpe={threshold_scores[BEST_THRESHOLD]:.3f} | 참고 AUC={_auc:.4f} | Val누적={_cum*100:.1f}%)\n')

In [ ]:
# =====================================================================
# 🚀 Cell 6: 최적 Threshold로 최종 패널 구성 + 워크포워드 피처 선택
# =====================================================================
print(f'🚀 2단계: threshold={BEST_THRESHOLD:.3f} 로 최종 패널 구성')

panel = build_panel(threshold=BEST_THRESHOLD)
feature_cols = [c for c in panel.columns if c not in EXCLUDE_COLS]
panel_norm = cross_sectional_normalize(panel, feature_cols)

train_mask, val_mask, test_mask = split_by_date(panel_norm)
print(f'📊 Train={train_mask.sum()}행 | Val={val_mask.sum()}행 | Test={test_mask.sum()}행 (유니버스 {len(UNIVERSE)}종목)')

# ✅ 워크포워드 피처 선택 — 거래일 순서로 정렬한 뒤 여러 확장 구간의 평균 중요도로 선택
#    (단일 스냅샷 importance보다 안정적)
train_sorted = panel_norm.loc[train_mask].sort_values('Date')
X_tr_sorted = train_sorted[feature_cols].values
y_tr_sorted = train_sorted['Label'].values

n_windows = 4
win_size = max(len(X_tr_sorted) // n_windows, 500)
importances = []
for w in range(n_windows):
    end = min(win_size * (w + 1), len(X_tr_sorted))
    rf_w = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_w.fit(X_tr_sorted[:end], y_tr_sorted[:end])
    importances.append(rf_w.feature_importances_)
avg_imp = np.mean(importances, axis=0)
feature_mask = avg_imp >= avg_imp.mean()
selected_features = [f for f, m in zip(feature_cols, feature_mask) if m]

print(f'✅ 피처 선택 완료: {len(feature_cols)}개 → {len(selected_features)}개')
print(f'   선택된 피처: {selected_features}')

X_tr = panel_norm.loc[train_mask, selected_features].values
y_tr = panel_norm.loc[train_mask, 'Label'].values
X_vl = panel_norm.loc[val_mask, selected_features].values
y_vl = panel_norm.loc[val_mask, 'Label'].values
X_te = panel_norm.loc[test_mask, selected_features].values
y_te = panel_norm.loc[test_mask, 'Label'].values

In [ ]:
# =====================================================================
# 🏋️ Cell 7: LightGBM(Optuna) + RandomForest 최종 학습 + 확률 캘리브레이션 + 저장
# =====================================================================
print('\n🔍 Optuna LightGBM 튜닝 중...')

def objective(trial):
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 300, 1000),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth'       : trial.suggest_int('max_depth', 4, 8),
        'num_leaves'      : trial.suggest_int('num_leaves', 20, 60),
        'subsample'       : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'class_weight': 'balanced', 'random_state': SEED, 'verbose': -1
    }
    m = lgb.LGBMClassifier(**params)
    m.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)],
          callbacks=[lgb.early_stopping(20, verbose=False)])
    return roc_auc_score(y_vl, m.predict_proba(X_vl)[:, 1])

# ✅ 재현성: TPESampler 시드 고정 (미고정 시 매 실행마다 다른 하이퍼파라미터가 선택됨)
study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED))
study.optimize(objective, n_trials=15, show_progress_bar=False)
lgbm_raw = lgb.LGBMClassifier(**study.best_params,
                              class_weight='balanced', random_state=SEED, verbose=-1)
lgbm_raw.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)],
             callbacks=[lgb.early_stopping(30, verbose=False)])
print(f'✅ LightGBM 완료! (Best AUC: {study.best_value:.4f})')

print('\n🌳 RandomForest 학습 중...')
rf_raw = RandomForestClassifier(
    n_estimators=500, max_depth=10,
    min_samples_split=10, min_samples_leaf=5,
    class_weight='balanced', random_state=SEED, n_jobs=-1
)
rf_raw.fit(X_tr, y_tr)
print('✅ RandomForest 완료!')

# ---------------------------------------------------------------------
# ✅ #15 확률 캘리브레이션 (isotonic) — 검증셋 기준.
#    sklearn>=1.6은 cv='prefit'가 제거되어 FrozenEstimator를 사용한다.
#    ※ 검증셋을 조기종료·가중치최적화와 공유하므로 val엔 다소 낙관적일 수 있음.
# ---------------------------------------------------------------------
def calibrate(estimator, X_cal, y_cal, method='isotonic'):
    try:
        from sklearn.frozen import FrozenEstimator          # sklearn >= 1.6
        cal = CalibratedClassifierCV(FrozenEstimator(estimator), method=method)
    except ImportError:                                     # 구버전 폴백
        try:
            cal = CalibratedClassifierCV(estimator=estimator, method=method, cv='prefit')
        except TypeError:
            cal = CalibratedClassifierCV(base_estimator=estimator, method=method, cv='prefit')
    cal.fit(X_cal, y_cal)
    return cal

if len(np.unique(y_vl)) > 1:
    lgbm_model = calibrate(lgbm_raw, X_vl, y_vl)
    rf_model   = calibrate(rf_raw,   X_vl, y_vl)
    auc_before = roc_auc_score(y_vl, lgbm_raw.predict_proba(X_vl)[:, 1])
    auc_after  = roc_auc_score(y_vl, lgbm_model.predict_proba(X_vl)[:, 1])
    print(f'✅ 확률 캘리브레이션 완료 (LGBM val AUC {auc_before:.4f} → {auc_after:.4f}, 랭킹 보존 확인용)')
else:
    lgbm_model, rf_model = lgbm_raw, rf_raw
    print('⚠️ 검증셋 라벨이 단일 클래스라 캘리브레이션 생략')

# =====================================================================
# 💾 모델 저장
# =====================================================================
save_dir = './models/cross_sectional_universe'
os.makedirs(save_dir, exist_ok=True)

with open(f'{save_dir}/lgbm.pkl', 'wb') as f:
    pickle.dump(lgbm_model, f)
with open(f'{save_dir}/rf.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

meta = {
    'UNIVERSE'         : UNIVERSE,
    'feature_cols'     : feature_cols,
    'selected_features': selected_features,
    'BEST_THRESHOLD'   : BEST_THRESHOLD,
    'PRED_DAYS'        : PRED_DAYS,
    'TRAIN_END_DATE'   : TRAIN_END_DATE,
    'VAL_END_DATE'     : VAL_END_DATE,
    'TRANSACTION_COST' : TRANSACTION_COST,
    'TOP_K'            : TOP_K,
    'SEED'             : SEED,
    'trained_at'       : datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),  # ✅ 실험추적
    'W_LGBM'           : W_LGBM,
    'W_RF'             : W_RF,
}
with open(f'{save_dir}/meta.pkl', 'wb') as f:
    pickle.dump(meta, f)

print(f'✅ 모델 저장 완료! → {save_dir}/')
print(f'   lgbm.pkl | rf.pkl | meta.pkl  (⚠️ pkl은 신뢰할 수 있는 자체 생성분만 로드할 것)')

In [ ]:
# =====================================================================
# ⚖️ Cell 8: Validation AUC 기준 앙상블 가중치 최적화 + 종목별 성능 점검
# =====================================================================
print('\n⚖️ Validation AUC 최대화 가중치 탐색 중 (scipy.optimize)...')

lgbm_prob_val  = lgbm_model.predict_proba(X_vl)[:, 1]
rf_prob_val    = rf_model.predict_proba(X_vl)[:, 1]
lgbm_prob_test = lgbm_model.predict_proba(X_te)[:, 1]
rf_prob_test   = rf_model.predict_proba(X_te)[:, 1]

def neg_val_auc(raw_w):
    w = np.abs(raw_w)
    w = w / (w.sum() + 1e-9)
    ens = w[0]*lgbm_prob_val + w[1]*rf_prob_val
    return -roc_auc_score(y_vl, ens)

opt_result = minimize(neg_val_auc, np.array([0.5, 0.5]), method='Nelder-Mead',
                       options={'xatol': 1e-4, 'fatol': 1e-4, 'maxiter': 500})
opt_w = np.abs(opt_result.x)
opt_w = opt_w / opt_w.sum()
W_LGBM, W_RF = opt_w

val_ens = W_LGBM*lgbm_prob_val + W_RF*rf_prob_val
val_auc = roc_auc_score(y_vl, val_ens)
val_auc_equal = roc_auc_score(y_vl, (lgbm_prob_val + rf_prob_val) / 2)
print(f'✅ W_LGBM={W_LGBM:.3f} | W_RF={W_RF:.3f}')
print(f'   Validation AUC (최적화 후): {val_auc:.4f}  (균등 가중치였다면: {val_auc_equal:.4f})')

final_test_ens = W_LGBM*lgbm_prob_test + W_RF*rf_prob_test
real_test_auc  = roc_auc_score(y_te, final_test_ens)

print("\n" + "="*55)
print(f"        🔥 [최종 검증] 유니버스 전체 크로스섹셔널 성능 🔥")
print("="*55)
print(f"   * 진짜 Test AUC : {real_test_auc:.4f}")
print("="*55)

# ✅ 종목별 Test AUC breakdown — 유니버스 모델이 특정 섹터/종목에 편중되지 않았는지 점검
test_panel = panel_norm.loc[test_mask].copy().reset_index(drop=True)
test_panel['pred_prob'] = final_test_ens

print('\n📊 종목별 Test AUC (상위 5개 / 하위 5개):')
per_ticker_auc = {}
for tk in UNIVERSE:
    sub = test_panel[test_panel['Ticker'] == tk]
    if len(sub) > 20 and sub['Label'].nunique() > 1:
        per_ticker_auc[tk] = roc_auc_score(sub['Label'], sub['pred_prob'])

sorted_auc = sorted(per_ticker_auc.items(), key=lambda x: -x[1])
for tk, auc in sorted_auc[:5]:
    print(f'   🔼 {tk}: {auc:.4f}')
print('   ...')
for tk, auc in sorted_auc[-5:]:
    print(f'   🔽 {tk}: {auc:.4f}')

with open(f'{save_dir}/meta.pkl', 'rb') as f:
    meta = pickle.load(f)
meta['W_LGBM'] = W_LGBM
meta['W_RF']   = W_RF
with open(f'{save_dir}/meta.pkl', 'wb') as f:
    pickle.dump(meta, f)
print(f'\n✅ meta.pkl 가중치 갱신 완료!')

In [ ]:
# =====================================================================
# 🚀 Cell 9: 크로스섹셔널 백테스트 — 상위 K종목 동일비중 롱
#   ✅ #4 거래비용 · #10 Sharpe/MDD 등 · #12 상승/하락 레짐별 성능 분해
# =====================================================================
print(f'\n🚀 백테스팅: 상위 {TOP_K}종목 동일비중 롱 (리밸런스={PRED_DAYS}영업일, 편도비용={TRANSACTION_COST*100:.2f}%)')

# 순수익(비용반영) / 총수익(비용무시) 둘 다 계산해 비용 영향 확인
portfolio_returns, market_returns, picked_tickers = crosssec_topk_backtest(
    test_panel, TOP_K, PRED_DAYS, cost=TRANSACTION_COST)
gross_returns, _, _ = crosssec_topk_backtest(
    test_panel, TOP_K, PRED_DAYS, cost=0.0)

cum_portfolio = np.cumprod(1 + portfolio_returns) - 1
cum_market    = np.cumprod(1 + market_returns) - 1

m_strat  = perf_metrics(portfolio_returns, pred_days=PRED_DAYS)
m_gross  = perf_metrics(gross_returns,     pred_days=PRED_DAYS)
m_market = perf_metrics(market_returns,    pred_days=PRED_DAYS)

print(f'   리밸런스 횟수: {m_strat["n"]}회')
print_metrics('전략(비용반영)', m_strat)
print_metrics('전략(비용무시)', m_gross)
print_metrics('시장(동일비중)', m_market)
print(f'   💸 거래비용이 갉아먹은 누적수익: {(m_gross["cum"]-m_strat["cum"])*100:.1f}%p')

# ---------------------------------------------------------------------
# #12 레짐 분해: 각 리밸런스를 '시장 상승기 vs 하락기'로 나눠 전략 성과 비교
#   → 하락장에서도 방어가 되는지(초과수익이 상승장 편중인지) 확인
# ---------------------------------------------------------------------
up_mask   = market_returns > 0
down_mask = ~up_mask
print('\n📉 레짐별 전략 성과 (시장 방향 기준):')
print_metrics(f'상승기({up_mask.sum()}회)',   perf_metrics(portfolio_returns[up_mask],   pred_days=PRED_DAYS))
print_metrics(f'하락기({down_mask.sum()}회)', perf_metrics(portfolio_returns[down_mask], pred_days=PRED_DAYS))

line_color = 'forestgreen' if cum_portfolio[-1] > cum_market[-1] else 'crimson'
plt.figure(figsize=(12, 5))
plt.plot(cum_market * 100, label='유니버스 전체 동일비중 (Buy & Hold)', color='gray', linestyle='--')
plt.plot((np.cumprod(1 + gross_returns) - 1) * 100, label=f'Top-{TOP_K} 전략 (비용무시)',
         color='lightgreen', linewidth=1, linestyle=':')
plt.plot(cum_portfolio * 100, label=f'Top-{TOP_K} 전략 (비용반영)', color=line_color, linewidth=2)
plt.title(f'크로스섹셔널 Top-{TOP_K} 롱 전략 백테스팅 (Test 구간, {VAL_END_DATE} ~ {END_DATE})')
plt.ylabel('Cumulative Return (%)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# =====================================================================
# 🔮 Cell 10: 실전 예측 — 유니버스 전체 랭킹 대시보드
#   ✅ #1 for_predict=True 로 add_features 호출 → 최신 거래일까지 보존.
#      (기존엔 학습용 add_features가 마지막 PRED_DAYS행을 잘라, 사실상
#       3일 전 데이터로 예측하면서 '오늘 기준'이라 표기하던 버그가 있었음.)
# =====================================================================
def predict_universe_today():
    print('📡 유니버스 전체 최신 데이터 수집 중...')
    ext_new = fetch_external_data(START_DATE, END_DATE)
    rows = []
    for tk in UNIVERSE:
        try:
            df_raw = fetch_us_data(tk, START_DATE, END_DATE)
            df_merged = df_raw.join(ext_new, how='left').ffill().dropna()
            df_feat = add_features(df_merged, threshold=BEST_THRESHOLD,
                                   pred_days=PRED_DAYS, for_predict=True)  # ✅ 최신일 보존
            df_feat['Ticker'] = tk
            rows.append(df_feat.iloc[[-1]])  # 최신 1행만
        except Exception as e:
            print(f'  ⚠️ {tk} 실패: {e}')

    latest = pd.concat(rows, axis=0)
    latest.index.name = 'Date'
    latest = latest.reset_index()

    # ✅ 오늘 하루치 유니버스 횡단면 기준으로 정규화 (학습 때와 동일한 방식)
    mean = latest[feature_cols].mean()
    std  = latest[feature_cols].std().replace(0, np.nan)
    z = ((latest[feature_cols] - mean) / std).clip(-5, 5).fillna(0.0)
    latest_norm = latest.copy()
    latest_norm[feature_cols] = z

    X_today = latest_norm[selected_features].values
    lgbm_p = lgbm_model.predict_proba(X_today)[:, 1]
    rf_p   = rf_model.predict_proba(X_today)[:, 1]
    prob   = W_LGBM * lgbm_p + W_RF * rf_p

    result = pd.DataFrame({
        'Ticker': latest_norm['Ticker'].values,
        'Close' : latest_norm['Close'].values,
        'Prob'  : prob,
    }).sort_values('Prob', ascending=False).reset_index(drop=True)

    result['Signal'] = result['Prob'].apply(
        lambda p: '🔥 BUY' if p > 0.5 + BEST_THRESHOLD
        else ('🚨 AVOID' if p < 0.5 - BEST_THRESHOLD else '💤 HOLD')
    )

    ref_date = latest_norm['Date'].iloc[0].strftime('%Y-%m-%d')
    print('\n' + '='*65)
    print(f' 🔮 크로스섹셔널 유니버스 랭킹 대시보드 ({ref_date} 기준)')
    print(f' 🎯 예측 타겟: 미래 {PRED_DAYS}영업일 뒤 방향성 (Threshold 마진: {BEST_THRESHOLD})')
    print('='*65)
    print(result.to_string(index=False))
    print('='*65)
    print(f' 🏆 상위 {TOP_K} BUY 후보: {", ".join(result.head(TOP_K)["Ticker"].tolist())}')
    print('='*65)
    return result

ranking_today = predict_universe_today()

In [ ]:
# =====================================================================
# 🔎 Cell 11 [진단]: Top-K 종목 빈도 분석 — 베타/모멘텀 쏠림인지 자동 판정 (#13)
# =====================================================================
from collections import Counter

pick_counts = Counter(picked_tickers)
total_picks = len(picked_tickers)
n_rebalance = len(portfolio_returns)

freq_df = pd.DataFrame([
    {'Ticker': tk, 'Count': cnt, 'Pct_of_rebalances': cnt / n_rebalance * 100}
    for tk, cnt in pick_counts.items()
]).sort_values('Count', ascending=False).reset_index(drop=True)

top5_share = freq_df.head(5)['Count'].sum() / total_picks * 100
print(f'📊 총 {n_rebalance}회 리밸런스 × Top-{TOP_K} = {total_picks}건의 선택 중, 등장 종목 수: {len(pick_counts)}개 (유니버스 {len(UNIVERSE)}개 중)')
print(f'   상위 5종목이 전체 선택의 {top5_share:.1f}%를 차지\n')
print(freq_df.head(10).to_string(index=False))

# ✅ #13 자동 경고: 균등 대비 쏠림이 심하면 '종목선정 능력'이 아니라 '소수 강세주 반복베팅' 의심
uniform_top5 = 100 * min(5, TOP_K) * 5 / (len(UNIVERSE))   # 대략적 균등 기대치(참고용)
print()
if top5_share > 70:
    print(f'🚨 [경고] 상위 5종목 집중도 {top5_share:.1f}% (>70%) — 초과수익이 소수 강세주에 대한')
    print('         반복 베팅일 가능성이 큼. 종목 선정력이라 단정하지 말 것.')
    print('         → 종목당 최대 비중 제한, 섹터 분산, 리밸런스 주기 조정 등을 검토하세요.')
elif top5_share > 50:
    print(f'⚠️ [주의] 상위 5종목 집중도 {top5_share:.1f}% (>50%) — 다소 편중. 분산 여부를 점검하세요.')
else:
    print(f'✅ [양호] 상위 5종목 집중도 {top5_share:.1f}% — 비교적 고르게 분산됨.')

plt.figure(figsize=(12, 5))
plt.bar(freq_df['Ticker'][:15], freq_df['Pct_of_rebalances'][:15], color='steelblue')
plt.axhline(y=100 * TOP_K / len(UNIVERSE), color='gray', linestyle='--',
            label=f'균등 선택 기준선 ({100*TOP_K/len(UNIVERSE):.1f}%)')
plt.ylabel('리밸런스 중 선택된 비율 (%)')
plt.title(f'Top-{TOP_K} 전략에서 종목별 선택 빈도 (상위 15개)')
plt.xticks(rotation=45)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# 🧪 Cell 12 [#4]: 저회전 변형 — 리밸런스 완화 + Top-K 확대로 거래비용 드래그 축소
#   목적: 알파 생성이 아니라 '거래비용/회전율 관리'.
#   기본 3일·Top5는 비용이 누적수익을 32%p나 갉아먹어 시장에 크게 졌음.
#   → 리밸런스 15일·Top10으로 회전율을 낮춰 '시장을 크게 지지 않는' 실전형 버전을 확인.
#   ※ 보유기간 수익은 라벨(3일 Future_Return)이 아니라 실제 종가(Close) 변화로 계산.
# =====================================================================
REBALANCE_DAYS = 15     # 3 → 15 (회전율↓)
TOP_K_LOW      = 10     # 5 → 10 (분산↑)

close_pivot = test_panel.pivot_table(index='Date', columns='Ticker', values='Close')
prob_pivot  = test_panel.pivot_table(index='Date', columns='Ticker', values='pred_prob')
dates = list(close_pivot.index)

port_r, mkt_r = [], []
prev = set()
turn_total = 0
for i in range(0, len(dates) - 1, REBALANCE_DAYS):
    d      = dates[i]
    j      = min(i + REBALANCE_DAYS, len(dates) - 1)
    d_next = dates[j]

    probs = prob_pivot.loc[d].dropna()
    if len(probs) == 0:
        continue
    picks = list(probs.sort_values(ascending=False).head(TOP_K_LOW).index)

    # 보유기간 실제 수익 (Close d → d_next)
    rets = []
    for tk in picks:
        c0, c1 = close_pivot.loc[d, tk], close_pivot.loc[d_next, tk]
        if pd.notna(c0) and pd.notna(c1) and c0 > 0:
            rets.append(c1 / c0 - 1)
    if not rets:
        continue
    gross = float(np.mean(rets))

    # 회전율 기반 비용: 교체된 종목 수 × 편도비용 / TOP_K
    new = set(picks)
    n_trade = len(new - prev) + len(prev - new)
    turn_total += n_trade
    cost = (n_trade / TOP_K_LOW) * TRANSACTION_COST
    port_r.append(gross - cost)
    prev = new

    # 같은 기간 시장(전체 동일비중) 수익
    mret = (close_pivot.loc[d_next] / close_pivot.loc[d] - 1).dropna()
    mkt_r.append(float(mret.mean()))

m_lo = perf_metrics(port_r, pred_days=REBALANCE_DAYS)
m_mk = perf_metrics(mkt_r, pred_days=REBALANCE_DAYS)

print(f'🧪 저회전 변형 (리밸런스 {REBALANCE_DAYS}일 · Top{TOP_K_LOW} · 비용반영):')
print(f'   리밸런스 {len(port_r)}회 | 누적 교체 {turn_total}건 (기본 3일·Top5 대비 회전율 대폭↓)')
print_metrics('저회전 전략', m_lo)
print_metrics('시장 동일비중', m_mk)
print()
gap = m_lo['cum'] - m_mk['cum']
if gap >= -0.05:
    print(f'✅ 시장과의 격차 {gap*100:+.1f}%p — 비용 관리로 "크게 지지 않는" 수준까지 회복.')
else:
    print(f'⚠️ 시장 대비 {gap*100:+.1f}%p — 여전히 열위. 이 피처셋엔 비용을 이길 알파가 없다는 방증.')
print('   ℹ️ 결론: 종목 선정으로 시장을 이기는 게 아니라 비용을 통제해 추종하는 수준. '
      '그럴 거면 단순 지수/동일비중 보유가 더 단순·견고 (FINDINGS.md 참조).')